# Dataset Inspection for CNN Fine-tuning

Run this before training to verify data loading, bbox crops, and
frame sampling are working correctly.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "cnn_finetune":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "cnn_finetune"))

from src.dataset import (
    CatBehaviorDataset,
    build_transforms,
    crop_frame_to_cat_bbox,
    extract_frames_from_video,
    get_frame_timestamps,
    load_manifest,
    resolve_video_path,
)
from src.model import CatPainCNN

CONFIG_PATH = PROJECT_ROOT / "cnn_finetune" / "config" / "default.yaml"
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

sns.set_theme(style="whitegrid")
CLASS_COLORS = {
    "Paining": "red",
    "Positive_Baseline": "green",
    "Agonistic": "orange",
    "Vocalizing": "purple",
    "HuntingMind": "blue",
    "Pain": "red",
    "No_Pain": "steelblue",
}
print(f"Project root: {PROJECT_ROOT}")
print(f"Config loaded: {CONFIG_PATH}")

In [ ]:
# ── Cell 3: Manifest overview ──
df = load_manifest(cfg)

print(f"\nTotal clips: {len(df)}")
has_video = df["video_path"].notna().sum()
print(f"Clips with video found: {has_video}")
print(f"Clips with video missing: {len(df) - has_video}")

# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts_5 = df["label_5"].value_counts().reindex(cfg["classes_5"])
colors_5 = [CLASS_COLORS[c] for c in counts_5.index]
counts_5.plot.bar(ax=axes[0], color=colors_5)
axes[0].set_title("5-Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts_5.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

counts_bin = df["label_binary"].value_counts()
colors_bin = [CLASS_COLORS.get(c, "gray") for c in counts_bin.index]
counts_bin.plot.bar(ax=axes[1], color=colors_bin)
axes[1].set_title("Binary Distribution")
axes[1].set_ylabel("Count")
for i, v in enumerate(counts_bin.values):
    axes[1].text(i, v + 1, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Also load predictions JSONL for display only
pred_path = PROJECT_ROOT / cfg.get("predictions_jsonl", "")
if pred_path.exists():
    pred_df = pd.read_json(pred_path, lines=True)
    print(f"\nAudio predictions JSONL: {len(pred_df)} rows (display only, not used for training)")
    display(pred_df.head(3))
else:
    print(f"\nPredictions JSONL not found at {pred_path}")

In [ ]:
# ── Cell 4: Frame extraction example ──
# Pick one clip from each of the 5 classes
try:
    from ultralytics import YOLO
    yolo_model = YOLO(str(PROJECT_ROOT / cfg.get("yolo_weights", "yolov8x.pt")))
    print("YOLO model loaded for bbox demo")
except Exception as e:
    yolo_model = None
    print(f"YOLO not available ({e}), showing raw frames only")

sample_clips = []
for cls in cfg["classes_5"]:
    subset = df[(df["label_5"] == cls) & df["video_path"].notna()]
    if len(subset) > 0:
        sample_clips.append(subset.iloc[0])

if sample_clips:
    fig, axes = plt.subplots(2, len(sample_clips) * 3, figsize=(5 * len(sample_clips), 7))
    if axes.ndim == 1:
        axes = axes.reshape(2, -1)

    col = 0
    for clip in sample_clips:
        ts = get_frame_timestamps(clip.get("duration_sec", 1.0), cfg.get("frames_per_clip", 3))
        frames, indices = extract_frames_from_video(clip["video_path"], ts)
        for fi, frame in enumerate(frames[:3]):
            # Row 1: raw
            axes[0, col].imshow(frame)
            axes[0, col].set_title(f"{clip['label_5']}\nt={ts[fi]:.2f}s", fontsize=8)
            axes[0, col].axis("off")
            # Row 2: bbox crop
            if yolo_model is not None:
                cropped, info = crop_frame_to_cat_bbox(frame, yolo_model, cfg)
                axes[1, col].imshow(cropped)
                bbox_str = f"bbox={'yes' if info['detected'] else 'no'}"
                if info['detected']:
                    bbox_str += f" conf={info['confidence']:.2f}"
                axes[1, col].set_title(bbox_str, fontsize=8)
            else:
                axes[1, col].imshow(frame)
                axes[1, col].set_title("no YOLO", fontsize=8)
            axes[1, col].axis("off")
            col += 1

    axes[0, 0].set_ylabel("Raw", fontsize=12)
    axes[1, 0].set_ylabel("YOLO crop", fontsize=12)
    plt.suptitle("Frame Extraction: 1 clip per class, 3 frames each", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No sample clips with video found")

In [ ]:
# ── Cell 5: Augmentation example ──
# Pick one Paining clip, show 3 frames x 4 augmented versions
pain_clips = df[(df["label_5"] == "Paining") & df["video_path"].notna()]
if len(pain_clips) > 0:
    clip = pain_clips.iloc[0]
    ts = get_frame_timestamps(clip.get("duration_sec", 1.0), 3)
    frames, _ = extract_frames_from_video(clip["video_path"], ts)

    train_tf = build_transforms(cfg, is_train=True)
    n_aug = 4

    if frames:
        fig, axes = plt.subplots(min(len(frames), 3), n_aug, figsize=(4 * n_aug, 4 * min(len(frames), 3)))
        if axes.ndim == 1:
            axes = axes.reshape(1, -1)

        for fi in range(min(len(frames), 3)):
            pil_img = Image.fromarray(frames[fi])
            for ai in range(n_aug):
                aug_tensor = train_tf(pil_img)
                # Unnormalize for display
                img = aug_tensor.clone()
                for c in range(3):
                    img[c] = img[c] * 0.229 + 0.485 if c == 0 else \
                             img[c] * 0.224 + 0.456 if c == 1 else \
                             img[c] * 0.225 + 0.406
                img = img.clamp(0, 1).permute(1, 2, 0).numpy()
                axes[fi, ai].imshow(img)
                axes[fi, ai].set_title(f"Frame {fi+1}, Aug {ai+1}", fontsize=9)
                axes[fi, ai].axis("off")

        plt.suptitle(f"Augmentation examples (Paining clip: {clip['stem']})", fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Could not extract frames from {clip['video_path']}")
else:
    print("No Paining clips with video found")

In [ ]:
# ── Cell 6: Class balance check across CV folds ──
labels = df["label_5_idx"].values
groups = df["cat_id"].values
binary_map = cfg["binary_map"]
classes_5 = cfg["classes_5"]

sgkf = StratifiedGroupKFold(
    n_splits=cfg["cv_folds"], shuffle=True,
    random_state=cfg.get("cv_random_state", 42),
)

print("CV Fold Composition:")
print(f"{'Fold':<6} | {'Train Pain':>10} | {'Train NoPain':>12} | {'Val Pain':>9} | {'Val NoPain':>11} | Warning")
print("-" * 75)

for fold_i, (train_idx, val_idx) in enumerate(sgkf.split(np.arange(len(df)), labels, groups), 1):
    train_labels = df.iloc[train_idx]["label_binary"]
    val_labels = df.iloc[val_idx]["label_binary"]
    t_pain = (train_labels == "Pain").sum()
    t_no = (train_labels == "No_Pain").sum()
    v_pain = (val_labels == "Pain").sum()
    v_no = (val_labels == "No_Pain").sum()
    warn = "\u26a0 NO PAIN IN VAL" if v_pain == 0 else ""
    print(f"  {fold_i:<4} | {t_pain:>10} | {t_no:>12} | {v_pain:>9} | {v_no:>11} | {warn}")

print("\nPer 5-class breakdown:")
for fold_i, (train_idx, val_idx) in enumerate(sgkf.split(np.arange(len(df)), labels, groups), 1):
    val_sub = df.iloc[val_idx]
    parts = [f"{cls[:4]}={int((val_sub['label_5'] == cls).sum())}" for cls in classes_5]
    print(f"  Fold {fold_i} val: {', '.join(parts)}")

In [ ]:
# ── Cell 7: Missing video report ──
missing = df[df["video_path"].isna()].copy()
print(f"Missing videos: {len(missing)} / {len(df)} ({100*len(missing)/len(df):.1f}%)")

if len(missing) > 0:
    print("\nMissing by class:")
    for cls in cfg["classes_5"]:
        total_cls = (df["label_5"] == cls).sum()
        miss_cls = (missing["label_5"] == cls).sum()
        pct = 100 * miss_cls / total_cls if total_cls > 0 else 0
        flag = "  \u26a0 WARNING" if pct > 20 else ""
        print(f"  {cls:<22s}: {miss_cls:>3d} / {total_cls:>3d} ({pct:>5.1f}%){flag}")
        if pct > 20:
            print(f"    \u26a0 {pct:.0f}% of {cls} clips have no video \u2014 results will be degraded")

    print("\nSample missing stems:")
    display(missing[["stem", "label_5", "label_binary"]].head(15))
else:
    print("All clips have video files found!")

In [ ]:
# ── Cell 8: Sample batch ──
from torch.utils.data import DataLoader

records = df.to_dict("records")
val_tf = build_transforms(cfg, is_train=False)
sample_ds = CatBehaviorDataset(records[:20], cfg, val_tf, is_train=False, yolo_model=None)

def collate_fn(batch):
    return {
        "frames": torch.stack([b["frames"] for b in batch]),
        "label_5": torch.stack([b["label_5"] for b in batch]),
        "label_binary": torch.stack([b["label_binary"] for b in batch]),
        "stem": [b["stem"] for b in batch],
    }

loader = DataLoader(sample_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
batch = next(iter(loader))

print(f"frames shape:       {batch['frames'].shape}")
print(f"label_5 values:     {batch['label_5'].tolist()}")
print(f"label_binary values: {batch['label_binary'].tolist()}")
print(f"stems:              {batch['stem']}")

# Quick model shape check with random init
model = CatPainCNN(cfg)
model.eval()
with torch.no_grad():
    out = model(batch["frames"])
print(f"\nModel output shapes:")
print(f"  logits_5:       {out['logits_5'].shape}")
print(f"  logits_binary:  {out['logits_binary'].shape}")
print(f"  frame_weights:  {out['frame_weights'].shape}")
print(f"  frame_weights:  {out['frame_weights'][0].tolist()} (random init, should sum to 1.0)")